# SALGINTR — Supplement S3, R analyses

**Physician-based digital participatory influenza-like-illness surveillance in Türkiye, season 2025/26 (2025-W40 to 2026-W20, 33 ISO weeks).**

This notebook runs every SALGINTR analysis whose estimator is a survival or mixed-effects model: the shared gamma-frailty Andersen-Gill recurrent-event models (M5 primary incidence, M6 vaccine effectiveness, M8 occupational exposure), the mixed-effects logistic models with adaptive Gauss-Hermite quadrature (M7b, both time directions), the ordinal proportional-odds model (M2), the binary logistic models with bootstrap optimism correction (M1, M3), the offset count models (M7a and the M8 recurrence companion), and the associated sensitivity analyses. The remaining SALGINTR analyses — descriptive, system-level concordance, aberration detection, selection and panel-validity work — are estimated in Python and are documented in the companion notebook.

## How to run this notebook

The workbook must sit **in the same folder as this notebook**:

```
SALGINTR_Unified_Dataset_20260809.xlsx
```

The first code cell sets `DATA` to that filename and reads every sheet from it with `readxl`. **If a later dated release supersedes this one, `DATA` is the single line to change** — nothing else in the notebook refers to the file. No paths, no placeholders: download the notebook and the workbook into one folder and run it unchanged.

## How the cells were executed

There is no Jupyter R kernel in this environment. The cells below are **R source**, executed as one shared R session by passing the concatenated source through `Rscript` (R 4.5.3, with `survival`, `MASS`, `nnet`, `lme4`, `geepack`, `pROC`, `readxl`). **The output shown under each cell is the real captured output of running that source** — stdout as R produced it, split back per cell. Nothing is transcribed by hand and nothing is illustrative. Cells share state and are therefore in dependency order: the panel built in the first cell, the primary fit in the second, and the objects each later cell reuses. The final cell prints the package versions the numbers were produced under.

## What this notebook does not do

- **No figure code.** Every analysis produces numeric output — model summaries, tables, printed statistics. Where a thesis figure derives from one of these analyses, the figure exists in the thesis and its plotting code is out of scope here.
- **No variable selection.** Every adjustment set is pre-specified and fixed on epidemiological grounds. No stepwise procedure, no data-driven screening, and no covariate is added or dropped in light of a fitted result. The descriptive covariate screen (cell R21) is printed to describe the covariate space and explicitly does not revise anything.
- **No imputation of declined responses.** Six physicians declined the item on health conditions conferring a risk of influenza complications. Those responses stay missing. This fixes the individual-model analysis sample at 242 physicians and the multivariable descriptive fit at 4,668 person-weeks.
- **Susceptibility is four levels throughout** (8 / 153 / 67 / 14 in the 242-physician frame), entered as an ordered score.
- **Stored weekly rate columns are never read.** The weekly incidence used anywhere in this notebook is computed as episodes divided by at-risk person-weeks; the stored rate columns follow a filed-week denominator instead.
- **Unweighted estimates are the estimates of record.** Weighting appears in the companion notebook as one sensitivity analysis, not as an estimation method.

## Each analysis block

Every analysis is a markdown cell of **English pseudocode** describing the procedure in words, followed by the R source cell and its captured output. The pseudocode is the specification; the code is one implementation of it.


---

# S3.1 Dataset and descriptive analysis

## R01 — Data access and person-week panel construction

*Analyses: A19 · Group S3.1*

**Pseudocode**

1. Set DATA to the workbook filename. The workbook must sit in the same folder as this notebook; if a later dated release supersedes this one, DATA is the single line to change.
2. Read the physician sheet, the person-week sheet, the episode sheet and the weekly system sheet with readxl.
3. Sort the person-week panel by physician identifier and then by week index, so that the counting-process intervals are in time order within each physician.
4. Derive, within each physician's own record: whether the week is that physician's first observed week, and whether a reporting gap immediately precedes the week.
5. Print the panel structure checks that every later block depends on: number of filed person-weeks, number of physicians, at-risk person-weeks, symptomatic weeks, continuation weeks (symptomatic but not a new onset), incident episodes, and the episode count in each of the four agent clusters.
6. Compute the season incidence on the AT-RISK denominator as 1000 times new episodes divided by at-risk person-weeks. The stored weekly rate columns follow the filed denominator instead and are never read.

In [1]:
suppressMessages({library(survival); library(MASS); library(nnet); library(lme4)
                  library(geepack); library(pROC)})
options(width = 200, stringsAsFactors = FALSE)

DATA <- "SALGINTR_Unified_Dataset_20260809.xlsx"

phys     <- as.data.frame(readxl::read_excel(DATA, sheet = "01_Physicians"))
pw       <- as.data.frame(readxl::read_excel(DATA, sheet = "02_PersonWeeks"))
episodes <- as.data.frame(readxl::read_excel(DATA, sheet = "03_Episodes"))
wsys     <- as.data.frame(readxl::read_excel(DATA, sheet = "04_WeeklySystem"))

pw <- pw[order(pw$participant_id, pw$week_idx), ]
pw$first_obs  <- unlist(lapply(split(pw$week_idx, pw$participant_id),
                               function(w) as.integer(seq_along(w) == 1)))
pw$prev_wk    <- ave(pw$week_idx,    pw$participant_id, FUN = function(w) c(NA, head(w, -1)))
pw$prev_sym   <- ave(pw$symptomatic, pw$participant_id, FUN = function(w) c(NA, head(w, -1)))
pw$gap_len    <- pw$week_idx - pw$prev_wk - 1
pw$ev_prior   <- ave(pw$event, pw$participant_id, FUN = function(x) cumsum(c(0, head(x, -1))))
pw$ever_vax   <- as.integer(!is.na(pw$vax_week_idx))
LASTWK        <- max(pw$week_idx)

cat("registered physicians                 :", nrow(phys), "\n")
cat("reporting physicians (>=1 weekly form):", sum(phys$cohort_248_reporter == 1), "\n")
cat("never-reporters                       :", sum(phys$cohort_248_reporter == 0), "\n")
cat("filed person-weeks                    :", nrow(pw), "\n")
cat("physicians in the panel               :", length(unique(pw$participant_id)), "\n")
cat("at-risk person-weeks                  :", sum(pw$at_risk_new_episode), "\n")
cat("symptomatic person-weeks              :", sum(pw$symptomatic), "\n")
cat("continuation weeks (sympt, not onset) :", sum(pw$symptomatic == 1 & pw$new_episode == 0), "\n")
cat("incident episodes                     :", sum(pw$new_episode), "\n")
cat("episodes with event indicator         :", sum(pw$event), "\n")
cat("agent clusters A/B/C/D                :",
    sum(pw$event_A), "/", sum(pw$event_B), "/", sum(pw$event_C), "/", sum(pw$event_D), "\n")
cat("weeks in the season                   :", LASTWK, "\n")
cat("forms filed in the final week         :", sum(pw$week_idx == LASTWK), "\n\n")

cat(sprintf("season incidence, at-risk denominator : %.5f episodes per 100 at-risk person-weeks\n",
            100 * sum(pw$event) / sum(pw$at_risk_new_episode)))
cat(sprintf("attack rate (>=1 episode)             : %.4f%%\n",
            100 * mean(phys$any_symptom[phys$cohort_248_reporter == 1] > 0)))

registered physicians                 : 304 
reporting physicians (>=1 weekly form): 248 
never-reporters                       : 56 
filed person-weeks                    : 4729 
physicians in the panel               : 248 
at-risk person-weeks                  : 4626 
symptomatic person-weeks              : 600 
continuation weeks (sympt, not onset) : 103 
incident episodes                     : 497 
episodes with event indicator         : 497 
agent clusters A/B/C/D                : 91 / 341 / 43 / 22 
weeks in the season                   : 33 
forms filed in the final week         : 157 

season incidence, at-risk denominator : 10.74362 episodes per 100 at-risk person-weeks
attack rate (>=1 episode)             : 77.4194%


---

# S3.2 Individual-level analysis

## R02 — M5 — primary ILI incidence, shared gamma-frailty Andersen-Gill

*Analyses: A28 · Group S3.2*

**Pseudocode**

1. Take the FULL filed person-week panel: 4,729 counting-process intervals from 248 physicians, 497 incident episodes. The at-risk restriction is not applied here; the panel-definition contrast is a separate sensitivity block.
2. Form the recurrent-event outcome as a counting-process survival object on the total-time clock: interval start, interval stop, and an event indicator that is one in the week an episode begins.
3. Fit a shared gamma-frailty Andersen-Gill model with a per-physician frailty term, Efron handling of ties, and the pre-specified covariate set fixed on epidemiological grounds: time-varying current-season vaccination, age in ten-year units, any household school-age child, and self-reported susceptibility in four ordered levels entered as a linear score. No variable selection of any kind is performed.
4. Print the model summary, then the four hazard ratios with Wald 95 per cent intervals and p-values, and the estimated frailty variance.
5. Confirm the frailty variance and the four hazard ratios against the values of record.

In [2]:
M5_RHS <- "vax_protected + age10 + school_kids_any + ili_freq_ord4"

fit_M5 <- coxph(as.formula(paste("Surv(tstart, tstop, event) ~", M5_RHS,
                                 "+ frailty(participant_id, distribution = 'gamma')")),
                data = pw, ties = "efron")
print(summary(fit_M5))

hr_table <- function(fit, k = NULL) {
  b <- fit$coefficients
  if (is.null(k)) k <- length(b)
  s <- sqrt(diag(fit$var))[seq_len(k)]
  round(cbind(HR = exp(b), lower95 = exp(b - 1.96 * s), upper95 = exp(b + 1.96 * s),
              se_logHR = s, p = 2 * pnorm(-abs(b / s))), 4)
}
cat("\nhazard ratios with Wald intervals\n")
print(hr_table(fit_M5))
cat(sprintf("\nfrailty variance theta = %.6f\n", fit_M5$history[[1]]$theta))
cat("person-weeks =", fit_M5$n, "  events =", fit_M5$nevent,
    "  physicians =", length(unique(pw$participant_id)), "\n")

Call:
coxph(formula = as.formula(paste("Surv(tstart, tstop, event) ~", 
    M5_RHS, "+ frailty(participant_id, distribution = 'gamma')")), 
    data = pw, ties = "efron")

  n= 4729, number of events= 497 

                          coef     se(coef) se2     Chisq  DF    p      
vax_protected             -0.04174 0.12727  0.10461   0.11  1.00 7.4e-01
age10                     -0.16095 0.06316  0.04948   6.49  1.00 1.1e-02
school_kids_any            0.30991 0.12952  0.09666   5.73  1.00 1.7e-02
ili_freq_ord4              0.30570 0.09951  0.07438   9.44  1.00 2.1e-03
frailty(participant_id, d                           162.02 90.59 6.0e-06

                exp(coef) exp(-coef) lower .95 upper .95
vax_protected      0.9591     1.0426    0.7474    1.2308
age10              0.8513     1.1746    0.7522    0.9635
school_kids_any    1.3633     0.7335    1.0577    1.7573
ili_freq_ord4      1.3576     0.7366    1.1170    1.6499

Iterations: 7 outer, 30 Newton-Raphson
     Variance of random effec

## R03 — M5 — frailty variance: profile-likelihood interval and likelihood-ratio test

*Analyses: A29 · Group S3.2*

**Pseudocode**

1. Refit the primary specification at a grid of fixed frailty variances, holding the variance at each grid value rather than letting it be estimated. Record the integrated (marginal) log-likelihood at each value.
2. Locate the maximum of the profile and report twice the drop from the maximum at each grid point.
3. Invert the profile at a drop of one half of the 95th percentile of the chi-square distribution on one degree of freedom to obtain the two interval bounds by root-finding.
4. Compute the likelihood-ratio statistic for the frailty term two ways, and print both, because they answer different questions: (a) the difference between the overall likelihood-ratio statistic of the frailty model and that of the frailty-free model on the same covariates, and (b) twice the difference between the integrated log-likelihood at the fitted variance and the partial log-likelihood of the frailty-free fit.
5. Report the p-value with the one-sided boundary correction appropriate to a variance tested at zero: half the nominal chi-square tail on one degree of freedom.

In [3]:
prof_ll <- function(theta) {
  f <- coxph(as.formula(paste("Surv(tstart, tstop, event) ~", M5_RHS,
             "+ frailty(participant_id, distribution = 'gamma', theta =", theta,
             ", sparse = FALSE)")), data = pw, ties = "efron")
  f$history[[1]]$c.loglik
}
theta_hat <- fit_M5$history[[1]]$theta
grid <- c(0.10, 0.15, 0.20, 0.25, 0.30, theta_hat, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70)
lls  <- sapply(grid, prof_ll)
lmax <- max(lls)
cat("profile of the integrated log-likelihood in the frailty variance\n")
print(round(cbind(theta = grid, loglik = lls, deviance_from_max = 2 * (lmax - lls)), 4))

drop_fn <- function(t) lmax - prof_ll(t) - qchisq(0.95, 1) / 2
lo <- uniroot(drop_fn, c(0.12, theta_hat))$root
hi <- uniroot(drop_fn, c(theta_hat, 0.90))$root
cat(sprintf("\ntheta = %.4f, profile-likelihood 95%% interval %.4f to %.4f\n", theta_hat, lo, hi))

fit_nofr <- coxph(as.formula(paste("Surv(tstart, tstop, event) ~", M5_RHS)),
                  data = pw, ties = "efron")
lr_overall <- 2 * diff(fit_M5$loglik) - 2 * diff(fit_nofr$loglik)
lr_direct  <- 2 * (prof_ll(theta_hat) - fit_nofr$loglik[2])
cat(sprintf("difference of overall likelihood-ratio statistics : chi-square = %.4f\n", lr_overall))
cat(sprintf("integrated vs partial log-likelihood              : chi-square = %.4f\n", lr_direct))
cat(sprintf("one-sided boundary-corrected p (direct form)      : %.3g\n",
            0.5 * pchisq(lr_direct, 1, lower.tail = FALSE)))

profile of the integrated log-likelihood in the frailty variance
       theta    loglik deviance_from_max
 [1,] 0.1000 -2416.117           15.8350
 [2,] 0.1500 -2412.485            8.5708
 [3,] 0.2000 -2410.214            4.0293
 [4,] 0.2500 -2408.906            1.4131
 [5,] 0.3000 -2408.302            0.2037
 [6,] 0.3332 -2408.200            0.0000
 [7,] 0.3500 -2408.224            0.0482
 [8,] 0.4000 -2408.548            0.6972
 [9,] 0.4500 -2409.184            1.9697
[10,] 0.5000 -2410.065            3.7308
[11,] 0.5500 -2411.139            5.8784
[12,] 0.6000 -2412.367            8.3339
[13,] 0.7000 -2415.167           13.9350

theta = 0.3332, profile-likelihood 95% interval 0.2027 to 0.5028
difference of overall likelihood-ratio statistics : chi-square = 245.5977
integrated vs partial log-likelihood              : chi-square = 44.8516
one-sided boundary-corrected p (direct form)      : 1.06e-11


## R04 — M6 — the vaccinated person-week partition under the two-week interval rule

*Analyses: A20 · Group S3.2*

**Pseudocode**

1. Restrict to person-weeks contributed by physicians who reported a current-season vaccination week.
2. Partition those weeks into three mutually exclusive and exhaustive groups: weeks at or after two weeks from the reported vaccination week (protected person-time, the exposed state); weeks inside the two-week interval, that is, at or after the vaccination week but not yet two weeks past it; and weeks preceding the vaccination week.
3. Check that the three parts sum to the total, print each count, express the interval weeks as a share of the vaccinated total, and express protected weeks as a share of the whole panel — this last share is the input to the minimum-detectable-effect calculation.

In [4]:
vx        <- pw[pw$ever_vax == 1, ]
protected <- vx$vax_protected == 1
interval  <- vx$vax_protected == 0 & vx$week_idx >= vx$vax_week_idx
before    <- vx$week_idx < vx$vax_week_idx

cat("person-weeks of vaccinated physicians  :", nrow(vx),
    " (", length(unique(vx$participant_id)), "physicians )\n")
cat("  protected person-time                :", sum(protected), "\n")
cat("  inside the two-week interval         :", sum(interval),
    sprintf(" (%.3f%% of the vaccinated weeks)\n", 100 * mean(interval)))
cat("  preceding the vaccination week       :", sum(before), "\n")
cat("  partition sums to the total          :", sum(protected) + sum(interval) + sum(before) == nrow(vx), "\n")
PROT_SHARE <- mean(pw$vax_protected)
cat(sprintf("protected share of the full 4,729-week panel : %.5f  (%.3f%%)\n",
            PROT_SHARE, 100 * PROT_SHARE))

person-weeks of vaccinated physicians  : 1874  ( 91 physicians )
  protected person-time                : 1613 
  inside the two-week interval         : 83  (4.429% of the vaccinated weeks)
  preceding the vaccination week       : 178 
  partition sums to the total          : TRUE 
protected share of the full 4,729-week panel : 0.34109  (34.109%)


## R05 — M6 — vaccine effectiveness, all-ILI and by agent cluster, with the B-cluster negative control

*Analyses: A32 · Group S3.2*

**Pseudocode**

1. Keep the M5 structure and specification unchanged; only the outcome changes. Vaccine effectiveness is one minus the hazard ratio on the time-varying protected-time indicator, expressed as a percentage, with the interval obtained by transforming the two hazard-ratio bounds.
2. Fit the same shared gamma-frailty Andersen-Gill model in turn to: all incident episodes (the pre-specified primary contrast); A-cluster episodes only, the viral-systemic influenza-like presentation (the agent-restricted secondary contrast); B-cluster episodes, the viral-local presentation, which serves as the negative control because an influenza vaccine should not act on it; and C- and D-cluster episodes for completeness.
3. For each outcome print the number of events, the hazard ratio with its interval and standard error on the log scale, the p-value, the effectiveness with its interval, and the frailty variance.
4. Read the B-cluster row as the negative control: an interval covering one is the expected result and supports the A-cluster contrast being agent-specific rather than a general artefact of who gets vaccinated.

In [5]:
ve_fit <- function(outcome, label) {
  d <- pw; d$.y <- outcome
  f <- coxph(as.formula(paste("Surv(tstart, tstop, .y) ~", M5_RHS,
             "+ frailty(participant_id, distribution = 'gamma')")), data = d, ties = "efron")
  b <- f$coefficients[["vax_protected"]]; s <- sqrt(diag(f$var))[1]
  cat(sprintf("%-24s events=%3d  HR=%.4f (%.4f-%.4f)  se(logHR)=%.4f  p=%.4f  VE=%5.1f%% (%.1f to %.1f)  theta=%.4f\n",
      label, sum(outcome), exp(b), exp(b - 1.96 * s), exp(b + 1.96 * s), s,
      2 * pnorm(-abs(b / s)), 100 * (1 - exp(b)),
      100 * (1 - exp(b + 1.96 * s)), 100 * (1 - exp(b - 1.96 * s)), f$history[[1]]$theta))
  c(b = b, se = s)
}
cat("time-varying vaccination contrast, shared gamma-frailty Andersen-Gill\n\n")
r_all <- ve_fit(pw$event,   "all-ILI (primary)")
r_A   <- ve_fit(pw$event_A, "A viral-systemic")
r_B   <- ve_fit(pw$event_B, "B viral-local (control)")
r_C   <- ve_fit(pw$event_C, "C cluster")
r_D   <- ve_fit(pw$event_D, "D cluster")

time-varying vaccination contrast, shared gamma-frailty Andersen-Gill

all-ILI (primary)        events=497  HR=0.9591 (0.7474-1.2308)  se(logHR)=0.1273  p=0.7429  VE=  4.1% (-23.1 to 25.3)  theta=0.3332
A viral-systemic         events= 91  HR=0.5368 (0.3118-0.9242)  se(logHR)=0.2772  p=0.0248  VE= 46.3% (7.6 to 68.8)  theta=0.5222
B viral-local (control)  events=341  HR=1.0783 (0.8065-1.4417)  se(logHR)=0.1482  p=0.6109  VE= -7.8% (-44.2 to 19.4)  theta=0.4152
C cluster                events= 43  HR=1.0794 (0.4487-2.5966)  se(logHR)=0.4479  p=0.8646  VE= -7.9% (-159.7 to 55.1)  theta=3.6011
D cluster                events= 22  HR=1.3795 (0.4803-3.9620)  se(logHR)=0.5383  p=0.5501  VE=-37.9% (-296.2 to 52.0)  theta=3.0870


## R06 — M6 — A-versus-B cluster contrast as a ratio of hazard ratios

*Analyses: A33 · Group S3.2*

**Pseudocode**

1. Take the two log hazard ratios for the protected-time indicator from the A-cluster and B-cluster fits, together with their standard errors.
2. Form the ratio of hazard ratios by subtracting the two log hazard ratios and exponentiating.
3. Obtain the standard error of the difference by the delta method for two independent estimates, that is the square root of the sum of the two squared standard errors, and build a Wald interval on the log scale before exponentiating.
4. Report the ratio, its interval and its p-value. A ratio below one with an interval excluding one says the vaccine association is stronger for the influenza-like cluster than for the negative-control cluster.

In [6]:
d_log  <- r_A["b"] - r_B["b"]
se_dif <- sqrt(r_A["se"]^2 + r_B["se"]^2)
cat(sprintf("ratio of hazard ratios (A vs B) = %.4f (%.4f-%.4f), p = %.4f  [delta-method se on the log scale = %.4f]\n",
    exp(d_log), exp(d_log - 1.96 * se_dif), exp(d_log + 1.96 * se_dif),
    2 * pnorm(-abs(d_log / se_dif)), se_dif))

ratio of hazard ratios (A vs B) = 0.4978 (0.2688-0.9218), p = 0.0265  [delta-method se on the log scale = 0.3143]


## R07 — M6 — exposed-event accounting for the vaccine contrast

*Analyses: A34 · Group S3.2*

**Pseudocode**

1. For each outcome definition, cross-tabulate events by the state of the time-varying exposure indicator in the week the event occurred.
2. Print, for each outcome, how many of its events fell in protected person-time and the total, since the exposed-event count — not the number of physicians and not the number of person-weeks — is what determines the precision of the vaccine contrast.

In [7]:
cat("events occurring in protected person-time\n")
for (nm in c("event", "event_A", "event_B", "event_C", "event_D")) {
  lab <- c(event = "all-ILI", event_A = "A cluster", event_B = "B cluster",
           event_C = "C cluster", event_D = "D cluster")[nm]
  cat(sprintf("  %-10s %3d of %3d exposed (%.1f%%)\n", lab,
      sum(pw[[nm]] == 1 & pw$vax_protected == 1), sum(pw[[nm]]),
      100 * sum(pw[[nm]] == 1 & pw$vax_protected == 1) / sum(pw[[nm]])))
}

events occurring in protected person-time
  all-ILI    164 of 497 exposed (33.0%)
  A cluster   21 of  91 exposed (23.1%)
  B cluster  122 of 341 exposed (35.8%)
  C cluster   13 of  43 exposed (30.2%)
  D cluster    8 of  22 exposed (36.4%)


---

# S3.5 Sensitivity analyses and other analyses

## R08 — M6 — the three minimum-detectable-effect quantities, achieved power, and events required

*Analyses: A35, A79 · Group S3.5*

**Pseudocode**

1. Fix the exposure allocation at the achieved share of protected person-time from the partition block. This is a design input, not an estimate.
2. Under Schoenfeld's approximation the standard error of the log hazard ratio is one over the square root of the number of events times the exposure share times one minus that share. At two-sided five per cent significance and eighty per cent power the smallest detectable log hazard ratio is minus the sum of the two normal quantiles times that standard error.
3. Compute THREE quantities that are distinct and must never be interchanged. First, the all-ILI DESIGN boundary, using the 497 all-ILI events. Second, the A-cluster DESIGN boundary, using the 91 A-cluster events — this is the boundary of record for the agent-restricted contrast. Third, the effect implied by the A-cluster fit's OWN observed standard error, which is larger than the design standard error because the frailty model spends information on the physician random effect; this third quantity is not a design boundary at all.
4. State explicitly where the observed A-cluster effectiveness sits relative to the A-cluster design boundary.
5. Compute the power the design actually had against assumed true effectiveness values, using both the design standard error and the observed one.
6. Compute the exaggeration a significant estimate would carry if the true effectiveness were thirty per cent, by simulating from the sampling distribution at that truth and keeping only the replicates that reach significance. Report the factor on the EFFECTIVENESS scale — the mean significant effectiveness divided by the true thirty per cent — and, separately, the factor on the log-hazard scale, since the two are different numbers and the effectiveness-scale factor is the one that describes what a reader of a significant estimate would take away.
7. Invert the same formula to give the number of events that eighty per cent power would require at smaller true effects.

In [8]:
z_pow <- qnorm(0.975) + qnorm(0.80)
se_schoenfeld <- function(E, p) sqrt(1 / (E * p * (1 - p)))
mdes <- function(E, p) { s <- se_schoenfeld(E, p); hr <- exp(-z_pow * s)
                         c(se = s, HR = hr, VE_pct = 100 * (1 - hr)) }

cat(sprintf("protected person-time share used as the design input : %.6f\n\n", PROT_SHARE))
b_all <- mdes(497, PROT_SHARE); b_A <- mdes(91, PROT_SHARE)
se_obs <- r_A["se"]; hr_obs_bound <- exp(-z_pow * se_obs)

cat("THREE DISTINCT QUANTITIES\n")
cat(sprintf("  1. all-ILI DESIGN boundary   (497 events) : se = %.5f  HR = %.5f  VE = %.3f%%\n",
            b_all["se"], b_all["HR"], b_all["VE_pct"]))
cat(sprintf("  2. A-cluster DESIGN boundary  (91 events) : se = %.5f  HR = %.5f  VE = %.3f%%   <- boundary of record\n",
            b_A["se"], b_A["HR"], b_A["VE_pct"]))
cat(sprintf("  3. effect implied by the A-cluster fit's OWN observed se = %.5f : HR = %.5f  VE = %.3f%%\n",
            se_obs, hr_obs_bound, 100 * (1 - hr_obs_bound)))
ve_obs <- 100 * (1 - exp(r_A["b"]))
cat(sprintf("\nobserved A-cluster VE = %.1f%% against the A-cluster design boundary of %.1f%% : observed %s the boundary\n",
            ve_obs, b_A["VE_pct"], ifelse(ve_obs >= b_A["VE_pct"], "sits on or beyond", "falls short of")))

cat("\nachieved power against assumed true effectiveness\n")
for (VE in c(0.30, 0.40, 0.50)) {
  bt <- log(1 - VE)
  cat(sprintf("  true VE = %2.0f%%  power with the design se = %.4f   power with the observed se = %.4f\n",
      100 * VE, pnorm(abs(bt) / b_A["se"] - qnorm(0.975)),
      pnorm(abs(bt) / se_obs - qnorm(0.975))))
}
bt30 <- log(1 - 0.30); crit <- qnorm(0.975) * se_obs
set.seed(20260707); draws <- rnorm(4e6, bt30, se_obs)
sig  <- abs(draws) > crit & draws < 0
cat(sprintf("  if the true effectiveness were 30%%, a significant estimate would average %.1f%% effectiveness\n",
    100 * (1 - exp(mean(draws[sig])))))
cat(sprintf("  exaggeration on the EFFECTIVENESS scale  = %.2f-fold\n",
    100 * (1 - exp(mean(draws[sig]))) / 30))
cat(sprintf("  exaggeration on the log-hazard scale     = %.2f-fold  (a different quantity; not interchangeable)\n",
    mean(draws[sig]) / bt30))

cat("\nevents required for 80% power at smaller true effects, same protected share\n")
cat("  (the plan of record carries 129 events at 40% and 266 at 30%; the recomputation below is a few per cent higher\n")
cat("   and the gap is not reproducible from the release, which does not record the exposure share those two used)\n")
for (VE in c(0.40, 0.30, 0.20)) {
  bt <- log(1 - VE); E <- (z_pow / bt)^2 / (PROT_SHARE * (1 - PROT_SHARE))
  cat(sprintf("  true VE = %2.0f%%  events required = %.1f  ->  %d\n", 100 * VE, E, ceiling(E)))
}

protected person-time share used as the design input : 0.341087

THREE DISTINCT QUANTITIES
  1. all-ILI DESIGN boundary   (497 events) : se = 0.09462  HR = 0.76714  VE = 23.286%
  2. A-cluster DESIGN boundary  (91 events) : se = 0.22112  HR = 0.53822  VE = 46.178%   <- boundary of record
  3. effect implied by the A-cluster fit's OWN observed se = 0.27721 : HR = 0.45995  VE = 54.005%

observed A-cluster VE = 46.3% against the A-cluster design boundary of 46.2% : observed sits on or beyond the boundary

achieved power against assumed true effectiveness
  true VE = 30%  power with the design se = 0.3643   power with the observed se = 0.2504
  true VE = 40%  power with the design se = 0.6369   power with the observed se = 0.4533
  true VE = 50%  power with the design se = 0.8799   power with the observed se = 0.7056
  if the true effectiveness were 30%, a significant estimate would average 50.8% effectiveness
  exaggeration on the EFFECTIVENESS scale  = 1.69-fold
  exaggeration on the log

---

# S3.2 Individual-level analysis

## R09 — M1 — current-season vaccination uptake, with optimism-corrected discrimination and calibration

*Analyses: A21, A22 · Group S3.2*

**Pseudocode**

1. Restrict to reporting physicians who answered the item on health conditions conferring a risk of influenza complications. Six physicians declined that item; those responses stay MISSING rather than being imputed to no, which fixes the analysis sample at 242 physicians.
2. Fit a binary logistic model for current-season vaccination on the pre-specified eight-term set: prior-season vaccination, age in ten-year units, sex, university affiliation, face-to-face patient examination, any health condition, any household school-age child, and susceptibility in four ordered levels. No variable selection.
3. Report odds ratios with Wald intervals and p-values, and the sample and event counts.
4. Compute the apparent area under the receiver operating characteristic curve.
5. Correct that area for optimism by the Efron bootstrap: draw one thousand samples with replacement from the analysis frame, refit the model in each replicate, and take the difference between the replicate's area on its own bootstrap sample and its area when applied to the original sample. The mean difference is the optimism; subtract it from the apparent value.
6. Assess calibration with the Hosmer-Lemeshow statistic over deciles of predicted risk on eight degrees of freedom, and with the bootstrap-corrected calibration slope, the apparent value of which is one by construction.

In [9]:
rep248 <- phys[phys$cohort_248_reporter == 1, ]
m1dat  <- rep248[!is.na(rep248$comp_risk_any_nan), ]
cat("reporting physicians :", nrow(rep248),
    "   with a health-condition response :", nrow(m1dat),
    "   declined (kept missing) :", sum(is.na(rep248$comp_risk_any_nan)), "\n")
cat("susceptibility levels in the analysis sample :",
    paste(table(m1dat$ili_freq_ord4), collapse = " / "), "\n\n")

M1_FORM <- vax ~ prev_vax + age10 + female + university + sees_pat + comp_risk_any_nan +
                 school_kids_any + ili_freq_ord4
fit_M1 <- glm(M1_FORM, data = m1dat, family = binomial)
or_table <- function(f) { b <- coef(f); s <- summary(f)$coefficients[, 2]
  round(cbind(OR = exp(b), lower95 = exp(b - 1.96 * s), upper95 = exp(b + 1.96 * s),
              p = 2 * pnorm(-abs(b / s))), 5) }
print(or_table(fit_M1))
cat("n =", length(fit_M1$fitted.values), "  vaccinated =", sum(fit_M1$y), "\n")

auc_of <- function(y, p) as.numeric(auc(roc(y, p, quiet = TRUE, direction = "<")))
hoslem <- function(y, p, g = 10) {
  q <- cut(p, quantile(p, seq(0, 1, length.out = g + 1)), include.lowest = TRUE)
  o <- tapply(y, q, sum); e <- tapply(p, q, sum); n <- tapply(y, q, length)
  X <- sum((o - e)^2 / (e * (1 - e / n)))
  c(chi_square = X, df = g - 2, p = pchisq(X, g - 2, lower.tail = FALSE))
}
efron_optimism <- function(form, data, B = 1000, seed = 20260707) {
  set.seed(seed); f <- glm(form, data = data, family = binomial)
  dd <- model.frame(f); n <- nrow(dd)
  app_auc <- auc_of(f$y, f$fitted.values); o_auc <- numeric(B); o_slope <- numeric(B)
  for (b in seq_len(B)) {
    fb <- suppressWarnings(glm(form, data = dd[sample(n, n, TRUE), ], family = binomial))
    o_auc[b]   <- auc_of(fb$y, fb$fitted.values) -
                  auc_of(dd[[1]], as.numeric(predict(fb, newdata = dd, type = "response")))
    lp         <- as.numeric(predict(fb, newdata = dd, type = "link"))
    o_slope[b] <- 1 - coef(suppressWarnings(glm(dd[[1]] ~ lp, family = binomial)))[2]
  }
  c(apparent_auc = app_auc, optimism_auc = mean(o_auc), corrected_auc = app_auc - mean(o_auc),
    corrected_slope = 1 - mean(o_slope))
}
cat("\ndiscrimination and calibration (Efron bootstrap, 1,000 replicates, seed 20260707)\n")
print(round(efron_optimism(M1_FORM, m1dat), 5))
cat("Hosmer-Lemeshow :\n"); print(round(hoslem(fit_M1$y, fit_M1$fitted.values), 4))

reporting physicians : 248    with a health-condition response : 242    declined (kept missing) : 6 
susceptibility levels in the analysis sample : 8 / 153 / 67 / 14 

                        OR lower95  upper95       p
(Intercept)        0.19453 0.03503  1.08037 0.06126
prev_vax          14.69884 7.41622 29.13291 0.00000
age10              0.98954 0.70504  1.38885 0.95153
female             0.80788 0.39464  1.65383 0.55945
university         0.42787 0.20663  0.88601 0.02226
sees_pat           0.82770 0.41517  1.65014 0.59114
comp_risk_any_nan  1.50324 0.65811  3.43369 0.33343
school_kids_any    1.94784 0.95866  3.95770 0.06529
ili_freq_ord4      1.06539 0.62006  1.83058 0.81858
n = 242   vaccinated = 90 

discrimination and calibration (Efron bootstrap, 1,000 replicates, seed 20260707)
   apparent_auc    optimism_auc   corrected_auc corrected_slope 
        0.84477         0.02467         0.82010         0.89946 
Hosmer-Lemeshow :
chi_square         df          p 
    1.5328     8.000

## R10 — M2 — susceptibility as a four-level proportional-odds model, with the assumption check

*Analyses: A23, A24, A25 · Group S3.2*

**Pseudocode**

1. Use the same 242-physician frame. The outcome is self-reported annual influenza-like-illness frequency in FOUR ordered levels, distributed 8 / 153 / 67 / 14. It is never treated as five levels.
2. Fit a proportional-odds ordinal logistic model on the pre-specified set: age in ten-year units, sex, any health condition, any household school-age child, household size, respiratory allergy, current smoking, and face-to-face patient examination. Report cumulative odds ratios with Wald intervals.
3. Test the proportional-odds assumption by a likelihood-ratio comparison against the unconstrained multinomial model on the same predictors, which relaxes the constraint that one set of coefficients applies across all cut-points. Degrees of freedom are the difference in fitted parameters.
4. Report model fit as McFadden's pseudo R-squared against the intercept-only ordinal model, and discrimination as the ordinal concordance over all discordant pairs of the linear predictor.
5. Test the hypothesis jointly across age, household school-age children, health condition and current smoking. Report the likelihood-ratio form against the model with those four terms removed, and the Wald form alongside, because the two do not have to agree numerically.

In [10]:
m2dat   <- m1dat
m2dat$y <- factor(m2dat$ili_freq_ord4, levels = 0:3, ordered = TRUE)
M2_RHS  <- "age10 + female + comp_risk_any_nan + school_kids_any + hh + allergy + smoker_current + sees_pat"
fit_M2  <- polr(as.formula(paste("y ~", M2_RHS)), data = m2dat, Hess = TRUE)
cf <- summary(fit_M2)$coefficients
k  <- length(fit_M2$coefficients); b <- cf[1:k, 1]; s <- cf[1:k, 2]
cat("outcome distribution across the four ordered levels :",
    paste(table(m2dat$y), collapse = " / "), "   n =", fit_M2$n, "\n\n")
print(round(cbind(cumulative_OR = exp(b), lower95 = exp(b - 1.96 * s),
                  upper95 = exp(b + 1.96 * s), p = 2 * pnorm(-abs(b / s))), 5))

fit_mn <- multinom(as.formula(paste("y ~", M2_RHS)), data = m2dat, trace = FALSE)
LR_po  <- deviance(fit_M2) - deviance(fit_mn)
df_po  <- fit_mn$edf - length(coef(fit_M2)) - length(fit_M2$zeta)
cat(sprintf("\nproportional-odds assumption, likelihood ratio against the unconstrained multinomial : chi-square = %.4f, df = %d, p = %.4f\n",
            LR_po, df_po, pchisq(LR_po, df_po, lower.tail = FALSE)))

fit_M2null <- polr(y ~ 1, data = m2dat, Hess = TRUE)
cat(sprintf("McFadden pseudo R-squared = %.5f\n", 1 - logLik(fit_M2) / logLik(fit_M2null)))
lp <- as.numeric(model.matrix(fit_M2)[, -1, drop = FALSE] %*% coef(fit_M2))
yy <- as.integer(m2dat$y); nc <- nd <- nt <- 0
for (i in 1:(length(yy) - 1)) for (j in (i + 1):length(yy)) {
  if (yy[i] == yy[j]) next
  sg <- sign(yy[i] - yy[j]) * sign(lp[i] - lp[j])
  if (sg > 0) nc <- nc + 1 else if (sg < 0) nd <- nd + 1 else nt <- nt + 1
}
cat(sprintf("ordinal concordance = %.5f  (concordant %d, discordant %d, tied %d)\n",
            (nc + 0.5 * nt) / (nc + nd + nt), nc, nd, nt))

joint <- c("age10", "school_kids_any", "comp_risk_any_nan", "smoker_current")
fit_M2red <- polr(y ~ female + hh + allergy + sees_pat, data = m2dat, Hess = TRUE)
LR_j <- deviance(fit_M2red) - deviance(fit_M2)
cat(sprintf("joint test, likelihood-ratio form : chi-square = %.4f, df = 4, p = %.5f\n",
            LR_j, pchisq(LR_j, 4, lower.tail = FALSE)))
V <- vcov(fit_M2); bb <- coef(fit_M2); idx <- match(joint, names(bb))
W <- as.numeric(t(bb[idx]) %*% solve(V[idx, idx]) %*% bb[idx])
cat(sprintf("joint test, Wald form             : chi-square = %.4f, df = 4, p = %.5f\n",
            W, pchisq(W, 4, lower.tail = FALSE)))

outcome distribution across the four ordered levels : 8 / 153 / 67 / 14    n = 242 

                  cumulative_OR lower95 upper95       p
age10                   0.69854 0.52978 0.92106 0.01100
female                  0.95939 0.53816 1.71031 0.88822
comp_risk_any_nan       1.80048 0.94207 3.44108 0.07517
school_kids_any         2.02498 0.99987 4.10109 0.05004
hh                      1.20271 0.88238 1.63931 0.24276
allergy                 1.13433 0.65868 1.95343 0.64948
smoker_current          0.56868 0.28456 1.13648 0.11008
sees_pat                0.84170 0.49318 1.43653 0.52749

proportional-odds assumption, likelihood ratio against the unconstrained multinomial : chi-square = 19.8079, df = 16, p = 0.2290
McFadden pseudo R-squared = 0.05334
ordinal concordance = 0.66240  (concordant 10066, discordant 5128, tied 9)
joint test, likelihood-ratio form : chi-square = 14.3665, df = 4, p = 0.00621
joint test, Wald form             : chi-square = 13.6228, df = 4, p = 0.00860


## R11 — M3 — prior-season care-seeking, with its eligibility note

*Analyses: A26, A27 · Group S3.2*

**Pseudocode**

1. Eligibility is reporting an influenza-like illness in the prior season; the outcome is whether that illness led to a presentation at a health facility. NOTE that eligibility and outcome derive from the same question on the registration questionnaire, so the outcome logically implies eligibility: nobody can have presented for a prior-season illness they did not report having. The eligible set is therefore not an independent restriction, and the model estimates who among those reporting an illness sought care.
2. Of 201 eligible physicians, 195 answered the health-condition item and enter the model; 53 sought care.
3. Fit a binary logistic model on the pre-specified set: age in ten-year units, sex, any health condition, susceptibility in four ordered levels, prior-season vaccination, current smoking and respiratory allergy. No variable selection.
4. Report odds ratios with intervals and p-values, the apparent area under the curve, the Efron bootstrap optimism correction and corrected calibration slope, the Hosmer-Lemeshow statistic, and the joint test across age, susceptibility, smoking, allergy and prior-season vaccination in both the likelihood-ratio and Wald forms.

In [11]:
elig <- rep248[rep248$had_ili_prev == 1, ]
m3dat <- elig[!is.na(elig$comp_risk_any_nan), ]
cat("reported a prior-season illness  :", nrow(elig), "\n")
cat("of those, answered the health-condition item (analysis sample) :", nrow(m3dat), "\n")
cat("sought care (events)             :", sum(m3dat$sought_care), "\n\n")

M3_FORM <- sought_care ~ age10 + female + comp_risk_any_nan + ili_freq_ord4 + prev_vax +
                         smoker_current + allergy
fit_M3 <- glm(M3_FORM, data = m3dat, family = binomial)
print(or_table(fit_M3))
cat("\ndiscrimination and calibration (Efron bootstrap, 1,000 replicates, seed 20260707)\n")
print(round(efron_optimism(M3_FORM, m3dat), 5))
cat("Hosmer-Lemeshow :\n"); print(round(hoslem(fit_M3$y, fit_M3$fitted.values), 4))

fit_M3red <- glm(sought_care ~ female + comp_risk_any_nan, data = m3dat, family = binomial)
LR3 <- fit_M3red$deviance - fit_M3$deviance
cat(sprintf("joint test, likelihood-ratio form : chi-square = %.4f, df = 5, p = %.6f\n",
            LR3, pchisq(LR3, 5, lower.tail = FALSE)))
b3 <- coef(fit_M3); V3 <- vcov(fit_M3)
i3 <- match(c("age10", "ili_freq_ord4", "smoker_current", "allergy", "prev_vax"), names(b3))
W3 <- as.numeric(t(b3[i3]) %*% solve(V3[i3, i3]) %*% b3[i3])
cat(sprintf("joint test, Wald form             : chi-square = %.4f, df = 5, p = %.6f\n",
            W3, pchisq(W3, 5, lower.tail = FALSE)))

reported a prior-season illness  : 201 
of those, answered the health-condition item (analysis sample) : 195 
sought care (events)             : 53 

                       OR lower95 upper95       p
(Intercept)       0.01786 0.00272 0.11738 0.00003
age10             1.41559 1.01200 1.98015 0.04239
female            1.49938 0.68949 3.26059 0.30681
comp_risk_any_nan 1.91038 0.83567 4.36722 0.12492
ili_freq_ord4     2.21847 1.28055 3.84337 0.00448
prev_vax          0.27900 0.12376 0.62895 0.00208
smoker_current    2.34109 1.01002 5.42635 0.04734
allergy           1.84459 0.90964 3.74052 0.08962

discrimination and calibration (Efron bootstrap, 1,000 replicates, seed 20260707)
   apparent_auc    optimism_auc   corrected_auc corrected_slope 
        0.74302         0.04403         0.69899         0.79214 
Hosmer-Lemeshow :
chi_square         df          p 
   14.6512     8.0000     0.0663 
joint test, likelihood-ratio form : chi-square = 25.6887, df = 5, p = 0.000103
joint test, Wald form 

## R12 — M8 — occupational exposure in the patient-facing subgroup, main effects only

*Analyses: A38, A39 · Group S3.2*

**Pseudocode**

1. Restrict the panel to the 112 physicians providing face-to-face patient examination with a daily patient-volume value.
2. Reduce to complete cases on the occupational and adjustment terms, which leaves 110 physicians, 1,724 at-risk person-weeks and 199 incident episodes.
3. Fit the same shared gamma-frailty Andersen-Gill structure with the four occupational main effects added to the pre-specified adjustment set: daily patient volume, aerosol-generating procedure share, monthly on-call shifts, consistent mask use, plus age in ten-year units, household school-age children and four-level susceptibility.
4. Fit MAIN EFFECTS ONLY. No interaction terms are estimable and none are fitted: tabulate mask use against aerosol-procedure share at physician level to show that one cell holds a single physician, which is why an interaction cannot be supported.
5. Report the seven hazard ratios with intervals, the frailty variance, and the joint four-degree-of-freedom Wald test across the four occupational terms.

In [12]:
occ_all <- pw[pw$cohort_112_primary == 1, ]
occ_vars <- c("pat_per_day", "aerosol_ord", "oncall_n", "mask",
              "age10", "school_kids_any", "ili_freq_ord4")
occ <- occ_all[complete.cases(occ_all[, occ_vars]), ]
cat("patient-facing subgroup      :", length(unique(occ_all$participant_id)), "physicians,",
    nrow(occ_all), "person-weeks,", sum(occ_all$at_risk_new_episode), "at risk,",
    sum(occ_all$event), "episodes\n")
cat("complete-case analysis set   :", length(unique(occ$participant_id)), "physicians,",
    nrow(occ), "person-weeks,", sum(occ$at_risk_new_episode), "at risk,",
    sum(occ$event), "episodes\n\n")

fit_M8 <- coxph(Surv(tstart, tstop, event) ~ pat_per_day + aerosol_ord + oncall_n + mask +
                age10 + school_kids_any + ili_freq_ord4 +
                frailty(participant_id, distribution = "gamma"), data = occ, ties = "efron")
print(hr_table(fit_M8, 7))
cat(sprintf("frailty variance theta = %.6f\n", fit_M8$history[[1]]$theta))

b8 <- fit_M8$coefficients[1:4]; V8 <- fit_M8$var[1:4, 1:4]
W8 <- as.numeric(t(b8) %*% solve(V8) %*% b8)
cat(sprintf("\njoint test of the four occupational terms : chi-square = %.4f, df = 4, p = %.4f\n",
            W8, pchisq(W8, 4, lower.tail = FALSE)))
cat("\nmask use by aerosol-generating procedure share, physician level\n")
print(table(mask = unique(occ[, c("participant_id", "mask", "aerosol_ord")])$mask,
            aerosol = unique(occ[, c("participant_id", "mask", "aerosol_ord")])$aerosol_ord))
cat("smallest cell holds",
    min(table(unique(occ[, c("participant_id","mask","aerosol_ord")])[, c("mask","aerosol_ord")])),
    "physician(s) — no interaction term is estimable and none is fitted\n")

patient-facing subgroup      : 112 physicians, 1788 person-weeks, 1746 at risk, 202 episodes
complete-case analysis set   : 110 physicians, 1766 person-weeks, 1724 at risk, 199 episodes

                    HR lower95 upper95 se_logHR      p
pat_per_day     1.0035  0.9939  1.0132   0.0049 0.4770
aerosol_ord     1.1602  0.9162  1.4693   0.1205 0.2175
oncall_n        0.9904  0.8964  1.0942   0.0509 0.8492
mask            0.8435  0.5388  1.3205   0.2287 0.4567
age10           0.8417  0.6895  1.0274   0.1017 0.0902
school_kids_any 1.8533  1.2243  2.8052   0.2115 0.0035
ili_freq_ord4   1.5513  1.1470  2.0981   0.1541 0.0044
frailty variance theta = 0.334291

joint test of the four occupational terms : chi-square = 1.9827, df = 4, p = 0.7389

mask use by aerosol-generating procedure share, physician level
    aerosol
mask  0  1  2  3
   0 45 17  1  4
   1 21 10  6  6
smallest cell holds 1 physician(s) — no interaction term is estimable and none is fitted


## R13 — M8 — recurrence count companion in the occupational subgroup

*Analyses: A40 · Group S3.2*

**Pseudocode**

1. Collapse the complete-case occupational panel to one row per physician, summing incident episodes and at-risk weeks and carrying the physician-level covariates.
2. Fit a negative-binomial regression of the episode count on the same specification with the logarithm of at-risk weeks as an offset, so the coefficients are incidence-rate ratios per at-risk week.
3. Report the incidence-rate ratios with intervals, the negative-binomial size parameter with its standard error, and the implied overdispersion as the reciprocal of the size.
4. Fit the corresponding Poisson model and print the likelihood-ratio statistic against it, to show that the extra dispersion parameter is doing work.

In [13]:
occ_ph <- aggregate(cbind(n_ep = event, atrisk = at_risk_new_episode) ~ participant_id,
                    data = occ, FUN = sum)
occ_ph <- merge(occ_ph, unique(occ[, c("participant_id", occ_vars)]), by = "participant_id")
cat("physicians :", nrow(occ_ph), "  episodes :", sum(occ_ph$n_ep),
    "  at-risk weeks :", sum(occ_ph$atrisk), "\n\n")
CNT_RHS <- "pat_per_day + aerosol_ord + oncall_n + mask + age10 + school_kids_any + ili_freq_ord4 + offset(log(atrisk))"
fit_nb8 <- glm.nb(as.formula(paste("n_ep ~", CNT_RHS)), data = occ_ph)
sm <- summary(fit_nb8); bb <- coef(fit_nb8); ss <- sm$coefficients[, 2]
print(round(cbind(IRR = exp(bb), lower95 = exp(bb - 1.96 * ss), upper95 = exp(bb + 1.96 * ss),
                  p = sm$coefficients[, 4]), 4))
cat(sprintf("\nnegative-binomial size = %.4f (se %.4f), overdispersion 1/size = %.4f\n",
            fit_nb8$theta, fit_nb8$SE.theta, 1 / fit_nb8$theta))
fit_po8 <- glm(as.formula(paste("n_ep ~", CNT_RHS)), data = occ_ph, family = poisson)
cat(sprintf("likelihood ratio, negative binomial against Poisson : chi-square = %.4f, p = %.3g\n",
            2 * as.numeric(logLik(fit_nb8) - logLik(fit_po8)),
            0.5 * pchisq(2 * as.numeric(logLik(fit_nb8) - logLik(fit_po8)), 1, lower.tail = FALSE)))

physicians : 110   episodes : 199   at-risk weeks : 1724 

                   IRR lower95 upper95      p
(Intercept)     0.0949  0.0355  0.2535 0.0000
pat_per_day     1.0029  0.9939  1.0120 0.5313
aerosol_ord     1.1317  0.8970  1.4279 0.2968
oncall_n        0.9959  0.9051  1.0959 0.9337
mask            0.8429  0.5550  1.2801 0.4227
age10           0.8449  0.6975  1.0236 0.0851
school_kids_any 1.8932  1.2800  2.8002 0.0014
ili_freq_ord4   1.4419  1.0791  1.9267 0.0133

negative-binomial size = 3.2821 (se 1.2466), overdispersion 1/size = 0.3047
likelihood ratio, negative binomial against Poisson : chi-square = 15.9464, p = 3.26e-05


---

# S3.4 Sampling, selection and panel-validity analysis

## R14 — M7b — informative observation, forward direction

*Analyses: A70 · Group S3.4*

**Pseudocode**

1. Build the forward risk set: every filed person-week that HAS a possible successor week in the season, that is, all 4,729 weeks less the 157 filed in the final week, which have no following week and so cannot contribute to the outcome. This leaves 4,572 person-weeks from 248 physicians.
2. Define the outcome as whether the same physician filed a report in the following week, by looking up that physician-week combination in the panel.
3. Cross-tabulate the outcome by whether the index week was symptomatic and report the two raw continuation percentages.
4. Report the crude pooled odds ratio ONLY to explain why it is not the quantity of record: it ignores that reporting persistence is a physician trait, so it mixes the within-physician contrast with the between-physician differences in overall diligence.
5. The estimate of record is the conditional one. Fit a mixed-effects logistic model with a per-physician random intercept and no other terms, using 30-node adaptive Gauss-Hermite quadrature, and report the conditional odds ratio with its interval, its p-value, and the random-intercept standard deviation.

In [14]:
fwd <- pw[pw$week_idx < LASTWK, ]
panel_key <- paste(pw$participant_id, pw$week_idx)
fwd$next_filed <- as.integer(paste(fwd$participant_id, fwd$week_idx + 1) %in% panel_key)
cat("forward risk set :", nrow(fwd), "person-weeks (", nrow(pw), "less the",
    sum(pw$week_idx == LASTWK), "filed in the final week ),",
    length(unique(fwd$participant_id)), "physicians\n\n")
tb <- table(symptomatic = fwd$symptomatic, filed_next_week = fwd$next_filed)
print(tb)
cat(sprintf("\nafter a symptomatic week  : %d of %d = %.4f%%\n",
            tb["1","1"], sum(tb["1",]), 100 * tb["1","1"] / sum(tb["1",])))
cat(sprintf("after an asymptomatic week: %d of %d = %.4f%%\n",
            tb["0","1"], sum(tb["0",]), 100 * tb["0","1"] / sum(tb["0",])))
crude <- (tb["1","1"] / tb["1","0"]) / (tb["0","1"] / tb["0","0"])
cat(sprintf("crude pooled odds ratio = %.4f  -- shown only to explain why it is not reported: it confounds the within-physician contrast with between-physician differences in reporting diligence\n\n", crude))

fit_fwd <- glmer(next_filed ~ symptomatic + (1 | participant_id), data = fwd,
                 family = binomial, nAGQ = 30, control = glmerControl(optimizer = "bobyqa"))
sf <- summary(fit_fwd); print(sf$coefficients)
bf <- fixef(fit_fwd)["symptomatic"]; sef <- sf$coefficients["symptomatic", "Std. Error"]
cat(sprintf("\nCONDITIONAL odds ratio (estimate of record) = %.4f (%.4f-%.4f), p = %.4f\n",
            exp(bf), exp(bf - 1.96 * sef), exp(bf + 1.96 * sef),
            sf$coefficients["symptomatic", "Pr(>|z|)"]))
cat(sprintf("random-intercept standard deviation = %.4f\n", sqrt(unlist(VarCorr(fit_fwd))[1])))

forward risk set : 4572 person-weeks ( 4729 less the 157 filed in the final week ), 248 physicians

           filed_next_week
symptomatic    0    1
          0  530 3451
          1   89  502

after a symptomatic week  : 502 of 591 = 84.9408%
after an asymptomatic week: 3451 of 3981 = 86.6868%
crude pooled odds ratio = 0.8663  -- shown only to explain why it is not reported: it confounds the within-physician contrast with between-physician differences in reporting diligence

              Estimate Std. Error    z value     Pr(>|z|)
(Intercept) 1.95922016  0.1367242 14.3297249 1.426848e-46
symptomatic 0.01621956  0.1520711  0.1066578 9.150605e-01

CONDITIONAL odds ratio (estimate of record) = 1.0164 (0.7544-1.3693), p = 0.9151
random-intercept standard deviation = 1.8313


## R15 — M7b — post-gap episode coding, backward direction

*Analyses: A71 · Group S3.4*

**Pseudocode**

1. Build the backward risk set: at-risk person-weeks that HAVE a preceding week in that physician's own record, that is, at-risk weeks less each physician's first observed week, for which no prior week exists and so no gap can be defined. This leaves 4,378 person-weeks from 232 physicians with 449 events.
2. The exposure is whether a reporting gap falls immediately before the week; the outcome is whether a symptomatic week is coded as a new onset rather than a continuation.
3. Cross-tabulate and report the two raw percentages coded incident.
4. Fit the same mixed-effects logistic model with a per-physician random intercept by 30-node adaptive Gauss-Hermite quadrature; report the conditional odds ratio with its Wald interval, its p-value, the profile-likelihood interval, and the random-intercept standard deviation.
5. Fit the marginal, population-averaged model by generalized estimating equations with an exchangeable working correlation clustered on physician, and report it alongside — the two answer different questions and the conditional estimate is the one of record.

In [15]:
bwd <- pw[pw$at_risk_new_episode == 1 & pw$first_obs == 0, ]
cat("backward risk set :", nrow(bwd), "person-weeks (at-risk weeks less each physician's first observed week ),",
    length(unique(bwd$participant_id)), "physicians,", sum(bwd$new_episode), "events\n\n")
tb2 <- table(gap_before = bwd$gap_before, coded_incident = bwd$new_episode)
print(tb2)
cat(sprintf("\npost-gap weeks  : %d of %d = %.4f%% coded incident\n",
            tb2["1","1"], sum(tb2["1",]), 100 * tb2["1","1"] / sum(tb2["1",])))
cat(sprintf("contiguous weeks: %d of %d = %.4f%% coded incident\n",
            tb2["0","1"], sum(tb2["0",]), 100 * tb2["0","1"] / sum(tb2["0",])))

fit_bwd <- glmer(new_episode ~ gap_before + (1 | participant_id), data = bwd,
                 family = binomial, nAGQ = 30, control = glmerControl(optimizer = "bobyqa"))
sb <- summary(fit_bwd); print(sb$coefficients)
bb2 <- fixef(fit_bwd)["gap_before"]; sb2 <- sb$coefficients["gap_before", "Std. Error"]
cat(sprintf("\nCONDITIONAL odds ratio (estimate of record) = %.4f (%.4f-%.4f), p = %.5f\n",
            exp(bb2), exp(bb2 - 1.96 * sb2), exp(bb2 + 1.96 * sb2),
            sb$coefficients["gap_before", "Pr(>|z|)"]))
cat(sprintf("random-intercept standard deviation = %.4f\n", sqrt(unlist(VarCorr(fit_bwd))[1])))
ci_prof <- suppressMessages(confint(fit_bwd, parm = "gap_before", method = "profile"))
cat(sprintf("profile-likelihood interval on the odds ratio = %.4f to %.4f\n",
            exp(ci_prof[1]), exp(ci_prof[2])))

bwd$pid <- factor(bwd$participant_id)
bwd <- bwd[order(bwd$pid, bwd$week_idx), ]
fit_gee <- geeglm(new_episode ~ gap_before, id = pid, data = bwd,
                  family = binomial, corstr = "exchangeable")
print(summary(fit_gee)$coefficients)
cat(sprintf("MARGINAL (population-averaged) odds ratio = %.4f\n", exp(coef(fit_gee)["gap_before"])))

backward risk set : 4378 person-weeks (at-risk weeks less each physician's first observed week ), 232 physicians, 449 events

          coded_incident
gap_before    0    1
         0 3484  366
         1  445   83

post-gap weeks  : 83 of 528 = 15.7197% coded incident
contiguous weeks: 366 of 3850 = 9.5065% coded incident
             Estimate Std. Error    z value      Pr(>|z|)
(Intercept) -2.406946 0.08402471 -28.645690 1.814016e-180
gap_before   0.509417 0.14565983   3.497306  4.699824e-04

CONDITIONAL odds ratio (estimate of record) = 1.6643 (1.2510-2.2142), p = 0.00047
random-intercept standard deviation = 0.7415
profile-likelihood interval on the odds ratio = 1.2449 to 2.2050
             Estimate   Std.err      Wald     Pr(>|W|)
(Intercept) -2.201619 0.0781063 794.53339 0.0000000000
gap_before   0.494872 0.1353621  13.36569 0.0002562696
MARGINAL (population-averaged) odds ratio = 1.6403


## R16 — M7b — onsets attributable to the episode-coding rule

*Analyses: A73 · Group S3.4*

**Pseudocode**

1. Under the linkage rule a symptomatic week is coded as a continuation only if the immediately preceding week was also filed and symptomatic. A gap therefore breaks the link, and an ongoing illness that spans an unfiled week is counted as a new onset.
2. Enumerate the onsets to which this can apply: onsets that follow a reporting gap AND whose last filed week before the gap was symptomatic. These are the onsets the rule creates, because linkage across the gap would have merged them into the earlier illness.
3. Report that count and its share of the 497 onsets, then the subset with a gap of exactly one week, which is the most plausible single continuing illness.
4. Print the distribution of gap length before those onsets so the reader can see how the count is composed.

In [16]:
ons <- pw[pw$new_episode == 1, ]
cat("total incident onsets :", nrow(ons), "\n")
cat("onsets falling on a post-gap week :", sum(ons$gap_before == 1), "\n\n")
attrib <- ons$gap_len >= 1 & ons$prev_sym == 1
cat(sprintf("onsets attributable to the coding rule (post-gap AND last filed week symptomatic) : %d of %d = %.2f%%\n",
            sum(attrib, na.rm = TRUE), nrow(ons), 100 * sum(attrib, na.rm = TRUE) / nrow(ons)))
one_wk <- ons$gap_len == 1 & ons$prev_sym == 1
cat(sprintf("  of which the gap is exactly one week : %d = %.2f%% of all onsets\n",
            sum(one_wk, na.rm = TRUE), 100 * sum(one_wk, na.rm = TRUE) / nrow(ons)))
cat("\ngap length before onsets whose last filed week was symptomatic\n")
print(table(gap_length = ons$gap_len[ons$prev_sym == 1], useNA = "ifany"))
cat("\ncontiguous onsets whose preceding week was symptomatic (coded new despite an adjacent symptomatic week) :",
    sum(ons$gap_len == 0 & ons$prev_sym == 1, na.rm = TRUE), "\n")

total incident onsets : 497 
onsets falling on a post-gap week : 83 

onsets attributable to the coding rule (post-gap AND last filed week symptomatic) : 22 of 497 = 4.43%
  of which the gap is exactly one week : 14 = 2.82% of all onsets

gap length before onsets whose last filed week was symptomatic
gap_length
   0    1    2    3   12 <NA> 
  76   14    5    2    1   48 

contiguous onsets whose preceding week was symptomatic (coded new despite an adjacent symptomatic week) : 76 


## R17 — M7a — reporting-intensity gradient in incidence

*Analyses: A67 · Group S3.4*

**Pseudocode**

1. Collapse the panel to one row per physician: incident episodes, at-risk weeks, and the number of weeks the physician reported, which equals the number of panel rows that physician contributes.
2. Fit a Poisson regression of the episode count on weeks reported per ten additional weeks, with age in ten-year units, household school-age children and four-level susceptibility, and the logarithm of at-risk weeks as an offset so coefficients read as incidence-rate ratios.
3. Because episode counts are clustered within physician and the Poisson variance assumption is not relied on, compute the sandwich (robust) variance directly from the model matrix, the fitted means and the response residuals, and build Wald intervals from it.
4. Report the adjusted gradient and the unadjusted gradient alongside.
5. A rate ratio below one means physicians who reported MORE weeks recorded FEWER episodes per at-risk week, which is the direction that matters for the validity limb: episode capture is denser among intermittent reporters.

In [17]:
ph_lvl <- aggregate(cbind(n_ep = event, atrisk = at_risk_new_episode) ~ participant_id,
                    data = pw, FUN = sum)
nwk <- aggregate(week_idx ~ participant_id, data = pw, FUN = length)
names(nwk)[2] <- "n_weeks"
ph_lvl <- merge(merge(ph_lvl, nwk, by = "participant_id"),
                unique(pw[, c("participant_id", "age10", "school_kids_any", "ili_freq_ord4")]),
                by = "participant_id")
ph_lvl$nwk10 <- ph_lvl$n_weeks / 10
cat("physicians :", nrow(ph_lvl), "  episodes :", sum(ph_lvl$n_ep),
    "  at-risk person-weeks :", sum(ph_lvl$atrisk), "\n\n")

robust_se <- function(fit) {
  X <- model.matrix(fit); mu <- fitted(fit); r <- residuals(fit, type = "response")
  bread <- solve(t(X) %*% (X * mu))
  sqrt(diag(bread %*% (t(X) %*% (X * r^2)) %*% bread))
}
irr_robust <- function(fit) { b <- coef(fit); s <- robust_se(fit)
  round(cbind(IRR = exp(b), lower95 = exp(b - 1.96 * s), upper95 = exp(b + 1.96 * s),
              robust_se = s, p = 2 * pnorm(-abs(b / s))), 5) }

GRAD_RHS <- "nwk10 + age10 + school_kids_any + ili_freq_ord4 + offset(log(atrisk))"
fit_grad <- glm(as.formula(paste("n_ep ~", GRAD_RHS)), data = ph_lvl, family = poisson)
cat("adjusted, Poisson with a log at-risk-week offset and robust variance\n")
print(irr_robust(fit_grad))
fit_grad_un <- glm(n_ep ~ nwk10 + offset(log(atrisk)), data = ph_lvl, family = poisson)
cat("\nunadjusted\n"); print(irr_robust(fit_grad_un))

physicians : 248   episodes : 497   at-risk person-weeks : 4626 

adjusted, Poisson with a log at-risk-week offset and robust variance
                    IRR lower95 upper95 robust_se       p
(Intercept)     0.22137 0.12356 0.39662   0.29752 0.00000
nwk10           0.80455 0.70954 0.91229   0.06412 0.00069
age10           0.85574 0.76115 0.96207   0.05976 0.00913
school_kids_any 1.28233 1.01740 1.61624   0.11807 0.03520
ili_freq_ord4   1.24463 1.05430 1.46930   0.08467 0.00975

unadjusted
                IRR lower95 upper95 robust_se       p
(Intercept) 0.18822 0.13962 0.25375   0.15241 0.00000
nwk10       0.79121 0.69567 0.89986   0.06565 0.00036


## R18 — M7a — incidence by engagement stratum

*Analyses: A68 · Group S3.4*

**Pseudocode**

1. Stratify physicians by how many weeks they reported into three engagement strata: most engaged, twenty-four weeks or more; moderately engaged, six to twenty-three weeks; least engaged, five weeks or fewer.
2. Within each stratum sum incident episodes and at-risk person-weeks and form the crude rate per one hundred at-risk person-weeks.
3. Attach exact Poisson intervals from the chi-square relation for a count, so the least engaged stratum with twelve episodes gets an honestly wide interval rather than a normal approximation.
4. Print the strata side by side with the season total, which must equal the incidence of record.

In [18]:
ph_lvl$stratum <- cut(ph_lvl$n_weeks, breaks = c(0, 5, 23, 33),
                      labels = c("least engaged (1-5 weeks)", "moderately engaged (6-23 weeks)",
                                 "most engaged (24-33 weeks)"))
tb <- data.frame(stratum    = levels(ph_lvl$stratum),
                 physicians = as.vector(table(ph_lvl$stratum)),
                 episodes   = as.vector(tapply(ph_lvl$n_ep,   ph_lvl$stratum, sum)),
                 at_risk    = as.vector(tapply(ph_lvl$atrisk, ph_lvl$stratum, sum)))
tb$rate_per100 <- 100 * tb$episodes / tb$at_risk
ci <- t(mapply(function(e, n) c(100 * qchisq(0.025, 2 * e) / 2 / n,
                                100 * qchisq(0.975, 2 * (e + 1)) / 2 / n),
               tb$episodes, tb$at_risk))
tb$lower95 <- ci[, 1]; tb$upper95 <- ci[, 2]
print(format(tb, digits = 5))
cat(sprintf("\nseason total : %.5f per 100 at-risk person-weeks (%d episodes / %d at-risk weeks)\n",
            100 * sum(tb$episodes) / sum(tb$at_risk), sum(tb$episodes), sum(tb$at_risk)))

                          stratum physicians episodes at_risk rate_per100 lower95 upper95
1       least engaged (1-5 weeks)         37       12      86     13.9535   7.210  24.374
2 moderately engaged (6-23 weeks)        106      216    1543     13.9987  12.194  15.995
3      most engaged (24-33 weeks)        105      269    2997      8.9756   7.935  10.115

season total : 10.74362 per 100 at-risk person-weeks (497 episodes / 4626 at-risk weeks)


---

# S3.5 Sensitivity analyses and other analyses

## R19 — M7a — overdispersion sensitivity for the gradient

*Analyses: A69 · Group S3.5*

**Pseudocode**

1. Refit the reporting-intensity gradient on exactly the same specification and offset as a negative-binomial regression, which estimates a dispersion parameter instead of relying on a robust variance to absorb extra-Poisson variation.
2. Report the incidence-rate ratios, the size parameter with its standard error, and the likelihood-ratio statistic against the Poisson fit.
3. This analysis has NO single scalar target of record; it is reported to show that the gradient does not depend on how dispersion is handled. Compare the gradient with the robust-variance Poisson estimate from the previous block.

In [19]:
fit_grad_nb <- glm.nb(as.formula(paste("n_ep ~", GRAD_RHS)), data = ph_lvl)
sg <- summary(fit_grad_nb); bg <- coef(fit_grad_nb); sgs <- sg$coefficients[, 2]
print(round(cbind(IRR = exp(bg), lower95 = exp(bg - 1.96 * sgs), upper95 = exp(bg + 1.96 * sgs),
                  p = sg$coefficients[, 4]), 5))
cat(sprintf("\nnegative-binomial size = %.4f (se %.4f)\n", fit_grad_nb$theta, fit_grad_nb$SE.theta))
cat(sprintf("likelihood ratio against Poisson : chi-square = %.4f\n",
            2 * as.numeric(logLik(fit_grad_nb) - logLik(fit_grad))))
cat(sprintf("gradient per ten more reported weeks : Poisson with robust variance %.4f, negative binomial %.4f\n",
            exp(coef(fit_grad)["nwk10"]), exp(coef(fit_grad_nb)["nwk10"])))
cat("this analysis carries no single scalar target of record; it is a dispersion sensitivity\n")

                    IRR lower95 upper95       p
(Intercept)     0.20499 0.10961 0.38335 0.00000
nwk10           0.82009 0.72011 0.93395 0.00279
age10           0.85948 0.76624 0.96408 0.00976
school_kids_any 1.32452 1.04997 1.67084 0.01772
ili_freq_ord4   1.25400 1.04917 1.49882 0.01286

negative-binomial size = 3.9163 (se 1.0363)
likelihood ratio against Poisson : chi-square = 28.9896
gradient per ten more reported weeks : Poisson with robust variance 0.8046, negative binomial 0.8201
this analysis carries no single scalar target of record; it is a dispersion sensitivity


## R20 — M7b — risk-set sensitivity of the post-gap odds ratio

*Analyses: A72 · Group S3.5*

**Pseudocode**

1. Refit the post-gap conditional model across four defensible risk-set definitions, changing only which person-weeks are eligible and leaving the estimator and the specification untouched.
2. The four are: at-risk weeks with each physician's first observed week dropped, which is the definition of record; at-risk weeks with no first-week exclusion; all filed weeks with the first observed week dropped; and at-risk weeks with both the first week and continuation weeks dropped.
3. For each, report the risk-set size, the number of physicians and events, the conditional odds ratio with its interval, and the random-intercept standard deviation.
4. The reported range across definitions is the sensitivity result. It should keep the same direction and significance throughout.

In [20]:
ctl <- glmerControl(optimizer = "bobyqa")
risk_sets <- list(
  "at-risk, first observed week dropped (of record)" = pw$at_risk_new_episode == 1 & pw$first_obs == 0,
  "at-risk, all weeks"                               = pw$at_risk_new_episode == 1,
  "all filed weeks, first observed week dropped"     = pw$first_obs == 0,
  "at-risk, first week and continuation weeks dropped" =
      pw$at_risk_new_episode == 1 & pw$first_obs == 0 & !(pw$symptomatic == 1 & pw$new_episode == 0))
ors <- numeric(0)
for (nm in names(risk_sets)) {
  d <- pw[risk_sets[[nm]], ]
  f <- glmer(new_episode ~ gap_before + (1 | participant_id), data = d, family = binomial,
             nAGQ = 30, control = ctl)
  sm <- summary(f); b <- fixef(f)["gap_before"]; s <- sm$coefficients["gap_before", "Std. Error"]
  ors <- c(ors, exp(b))
  cat(sprintf("%-52s n=%4d phys=%3d events=%3d  OR=%.4f (%.4f-%.4f) p=%.5f  sigma_u=%.4f\n",
      nm, nrow(d), length(unique(d$participant_id)), sum(d$new_episode),
      exp(b), exp(b - 1.96 * s), exp(b + 1.96 * s),
      sm$coefficients["gap_before", "Pr(>|z|)"], sqrt(unlist(VarCorr(f))[1])))
}
cat(sprintf("\nrange of the conditional odds ratio across the four risk sets : %.4f to %.4f\n",
            min(ors), max(ors)))

at-risk, first observed week dropped (of record)     n=4378 phys=232 events=449  OR=1.6643 (1.2510-2.2142) p=0.00047  sigma_u=0.7415
at-risk, all weeks                                   n=4626 phys=248 events=497  OR=1.5266 (1.1516-2.0237) p=0.00326  sigma_u=0.7478
all filed weeks, first observed week dropped         n=4481 phys=233 events=449  OR=1.7636 (1.3312-2.3364) p=0.00008  sigma_u=0.6699
at-risk, first week and continuation weeks dropped   n=4378 phys=232 events=449  OR=1.6643 (1.2510-2.2142) p=0.00047  sigma_u=0.7415

range of the conditional odds ratio across the four risk sets : 1.5266 to 1.7636


## R21 — M5 — descriptive covariate screen (reported for transparency, NOT model selection)

*Analyses: A30 · Group S3.5*

**Pseudocode**

1. This block exists to describe the covariate space, NOT to choose covariates. The adjustment set of the primary model was fixed in advance on epidemiological grounds and is not changed in light of anything printed here.
2. Fit each candidate covariate on its own in an Andersen-Gill model with a physician-clustered robust variance, and print the hazard ratio with its interval and p-value.
3. Then fit the multivariable descriptive model, whose complete-case sample of 4,668 person-weeks reflects the six declined health-condition responses being kept missing rather than imputed.
4. Report the social-contact score row explicitly: it is not in the primary adjustment set and its interval covers one.

In [21]:
cands <- c("ili_freq_ord4", "age10", "school_kids_any", "social_risk_score", "vax_protected",
           "female", "comp_risk_any_nan", "hh", "sees_pat", "university", "smoker_current", "allergy")
cat("univariable Andersen-Gill fits, physician-clustered robust variance\n")
cat("(descriptive only; the pre-specified adjustment set is not changed on the basis of these)\n\n")
for (v in cands) {
  f <- coxph(as.formula(paste("Surv(tstart, tstop, event) ~", v, "+ cluster(participant_id)")),
             data = pw, ties = "efron")
  b <- f$coefficients[1]; s <- sqrt(diag(f$var))[1]
  cat(sprintf("  %-19s HR = %.4f (%.4f-%.4f)  p = %.4f   person-weeks = %d\n",
      v, exp(b), exp(b - 1.96 * s), exp(b + 1.96 * s), 2 * pnorm(-abs(b / s)), f$n))
}
cat("\nmultivariable descriptive fit, complete case on the health-condition item\n")
fit_scr <- coxph(Surv(tstart, tstop, event) ~ age10 + school_kids_any + ili_freq_ord4 +
                 comp_risk_any_nan + cluster(participant_id), data = pw, ties = "efron")
cat("person-weeks =", fit_scr$n, "  events =", fit_scr$nevent, "\n")
print(hr_table(fit_scr))
cat("\nwith the social-contact score instead of the health-condition item\n")
fit_scr2 <- coxph(Surv(tstart, tstop, event) ~ age10 + school_kids_any + ili_freq_ord4 +
                  social_risk_score + cluster(participant_id), data = pw, ties = "efron")
cat("person-weeks =", fit_scr2$n, "  events =", fit_scr2$nevent, "\n")
print(hr_table(fit_scr2))

univariable Andersen-Gill fits, physician-clustered robust variance
(descriptive only; the pre-specified adjustment set is not changed on the basis of these)

  ili_freq_ord4       HR = 1.3933 (1.1614-1.6716)  p = 0.0004   person-weeks = 4729
  age10               HR = 0.8547 (0.7533-0.9698)  p = 0.0148   person-weeks = 4729
  school_kids_any     HR = 1.3179 (1.0267-1.6917)  p = 0.0303   person-weeks = 4729
  social_risk_score   HR = 0.9059 (0.7244-1.1329)  p = 0.3864   person-weeks = 4729
  vax_protected       HR = 0.9722 (0.7412-1.2751)  p = 0.8386   person-weeks = 4729
  female              HR = 1.1926 (0.9195-1.5469)  p = 0.1844   person-weeks = 4729
  comp_risk_any_nan   HR = 0.9096 (0.6692-1.2363)  p = 0.5450   person-weeks = 4668
  hh                  HR = 1.0588 (0.9533-1.1760)  p = 0.2858   person-weeks = 4729
  sees_pat            HR = 1.1375 (0.8769-1.4756)  p = 0.3319   person-weeks = 4729
  university          HR = 0.8290 (0.6430-1.0688)  p = 0.1479   person-weeks = 4729
 

## R22 — M5 — estimator-family comparison on one specification

*Analyses: A31 · Group S3.5*

**Pseudocode**

1. Hold the specification fixed and vary only the estimator, so that any movement in the four effects is attributable to the estimator family and not to a change of model.
2. The six families are: the shared gamma-frailty Andersen-Gill model of record; Andersen-Gill with a physician-clustered robust variance instead of a frailty; the Prentice-Williams-Peterson model, stratified by episode order on the total-time clock with the order capped at the fourth episode; a Poisson regression at physician level with a log at-risk-week offset; a negative-binomial regression with the same offset; and a Cox model on the first episode only, which discards recurrences.
3. For the two count families the time-varying vaccination indicator cannot be carried week by week, so it enters as the physician's share of protected weeks. Note this: the vaccine column is not strictly comparable across the six rows, while age, household children and susceptibility are.
4. Print the four effects for each family and the range across families.
5. Report the Prentice-Williams-Peterson row in detail, because conditioning on episode order removes the between-physician contrast that the frailty model retains and so attenuates the household-children and susceptibility effects.

In [22]:
fam <- list()
fam[["shared gamma-frailty AG (of record)"]] <- fit_M5$coefficients[1:4]
fit_rob <- coxph(as.formula(paste("Surv(tstart, tstop, event) ~", M5_RHS, "+ cluster(participant_id)")),
                 data = pw, ties = "efron")
fam[["AG, clustered robust variance"]] <- fit_rob$coefficients
pw$strt <- pmin(pw$ev_prior + 1, 4)
g0 <- ave(pw$tstart, pw$participant_id, pw$strt, FUN = min)
fit_pwp <- coxph(Surv(pw$tstart - g0, pw$tstop - g0, pw$event) ~ vax_protected + age10 +
                 school_kids_any + ili_freq_ord4 + strata(strt) + cluster(participant_id),
                 data = pw, ties = "efron")
fam[["Prentice-Williams-Peterson"]] <- fit_pwp$coefficients
ph_lvl$vax_share <- aggregate(vax_protected ~ participant_id, pw, sum)$vax_protected / ph_lvl$n_weeks
CNT2 <- "vax_share + age10 + school_kids_any + ili_freq_ord4 + offset(log(atrisk))"
fit_po2 <- glm(as.formula(paste("n_ep ~", CNT2)), data = ph_lvl, family = poisson)
fam[["Poisson, at-risk-week offset"]] <- coef(fit_po2)[-1]
fit_nb2 <- glm.nb(as.formula(paste("n_ep ~", CNT2)), data = ph_lvl)
fam[["negative binomial, same offset"]] <- coef(fit_nb2)[-1]
first_ep <- do.call(rbind, lapply(split(pw, pw$participant_id), function(g) {
  k <- which(g$event == 1)[1]; if (is.na(k)) k <- nrow(g); g[seq_len(k), ] }))
fit_first <- coxph(as.formula(paste("Surv(tstart, tstop, event) ~", M5_RHS)),
                   data = first_ep, ties = "efron")
fam[["Cox, first episode only"]] <- fit_first$coefficients

M <- t(sapply(fam, function(b) exp(b[1:4])))
colnames(M) <- c("vaccine", "age", "household children", "susceptibility")
print(round(M, 4))
cat("\nrange across the six families\n"); print(round(apply(M, 2, range), 4))
cat("\nnote: in the two count families the time-varying vaccination indicator enters as the physician's share of protected weeks, so the vaccine column is not strictly comparable across rows\n")
cat("\nPrentice-Williams-Peterson detail\n")
b <- fit_pwp$coefficients; s <- sqrt(diag(fit_pwp$var))
print(round(cbind(HR = exp(b), lower95 = exp(b - 1.96 * s), upper95 = exp(b + 1.96 * s),
                  p = 2 * pnorm(-abs(b / s))), 4))

                                    vaccine    age household children susceptibility
shared gamma-frailty AG (of record)  0.9591 0.8513             1.3633         1.3576
AG, clustered robust variance        0.9541 0.8513             1.2917         1.3098
Prentice-Williams-Peterson           0.9021 0.8733             1.2828         1.2262
Poisson, at-risk-week offset         0.9420 0.8580             1.2969         1.2723
negative binomial, same offset       0.9245 0.8611             1.3704         1.2950
Cox, first episode only              0.7300 0.8179             1.4207         1.3361

range across the six families
     vaccine    age household children susceptibility
[1,]  0.7300 0.8179             1.2828         1.2262
[2,]  0.9591 0.8733             1.4207         1.3576

note: in the two count families the time-varying vaccination indicator enters as the physician's share of protected weeks, so the vaccine column is not strictly comparable across rows

Prentice-Williams-Peterson

## R23 — Sensitivity — the two-week interval grid and the panel-definition comparison

*Analyses: A20, A28 · Group S3.5*

**Pseudocode**

1. For the interval sensitivity, rebuild the time-varying exposure indicator with the delay between the reported vaccination week and the start of protected time set to zero, one, two, three and four weeks in turn. Two weeks is the definition of record; the rest show whether the contrast depends on that choice.
2. At each delay report the number of exposed person-weeks and refit both the A-cluster and the all-ILI contrasts with the specification unchanged.
3. For the panel-definition comparison, refit the primary model on the full filed panel of 4,729 weeks (the definition of record) and then on at-risk weeks only, 4,626. The at-risk restriction removes the 103 continuation weeks, which carry no onset but do carry exposure and covariate time.
4. Print both fits and their frailty variances so the reader can see what the restriction costs.

In [23]:
cat("two-week interval sensitivity: delay from the reported vaccination week to protected time\n\n")
for (dl in 0:4) {
  vp <- as.integer(!is.na(pw$vax_week_idx) & pw$week_idx >= pw$vax_week_idx + dl)
  d <- pw; d$vp <- vp
  fa <- coxph(Surv(tstart, tstop, event_A) ~ vp + age10 + school_kids_any + ili_freq_ord4 +
              frailty(participant_id, distribution = "gamma"), data = d, ties = "efron")
  fl <- coxph(Surv(tstart, tstop, event) ~ vp + age10 + school_kids_any + ili_freq_ord4 +
              frailty(participant_id, distribution = "gamma"), data = d, ties = "efron")
  ba <- fa$coefficients[1]; sa <- sqrt(diag(fa$var))[1]; bl <- fl$coefficients[1]
  cat(sprintf("delay = %d week(s)%s  exposed weeks = %4d   A-cluster HR = %.4f (%.4f-%.4f) VE = %5.1f%%   all-ILI HR = %.4f VE = %4.1f%%\n",
      dl, ifelse(dl == 2, " [of record]", "           "), sum(vp),
      exp(ba), exp(ba - 1.96 * sa), exp(ba + 1.96 * sa), 100 * (1 - exp(ba)),
      exp(bl), 100 * (1 - exp(bl))))
}
cat("\npanel-definition comparison\n\n")
for (nm in c("full filed panel (of record)", "at-risk person-weeks only")) {
  d <- if (grepl("full", nm)) pw else pw[pw$at_risk_new_episode == 1, ]
  f <- coxph(as.formula(paste("Surv(tstart, tstop, event) ~", M5_RHS,
             "+ frailty(participant_id, distribution = 'gamma')")), data = d, ties = "efron")
  cat(sprintf("%-30s person-weeks = %4d  events = %3d  theta = %.4f\n",
      nm, nrow(d), sum(d$event), f$history[[1]]$theta))
  print(hr_table(f, 4))
  cat("\n")
}

two-week interval sensitivity: delay from the reported vaccination week to protected time

delay = 0 week(s)             exposed weeks = 1696   A-cluster HR = 0.5029 (0.2928-0.8637) VE =  49.7%   all-ILI HR = 0.9947 VE =  0.5%
delay = 1 week(s)             exposed weeks = 1658   A-cluster HR = 0.5202 (0.3025-0.8945) VE =  48.0%   all-ILI HR = 0.9827 VE =  1.7%
delay = 2 week(s) [of record]  exposed weeks = 1613   A-cluster HR = 0.5368 (0.3118-0.9242) VE =  46.3%   all-ILI HR = 0.9591 VE =  4.1%
delay = 3 week(s)             exposed weeks = 1562   A-cluster HR = 0.5162 (0.2972-0.8967) VE =  48.4%   all-ILI HR = 0.9584 VE =  4.2%
delay = 4 week(s)             exposed weeks = 1511   A-cluster HR = 0.5324 (0.3062-0.9257) VE =  46.8%   all-ILI HR = 0.9676 VE =  3.2%

panel-definition comparison

full filed panel (of record)   person-weeks = 4729  events = 497  theta = 0.3332
                    HR lower95 upper95 se_logHR      p
vax_protected   0.9591  0.7474  1.2308   0.1273 0.7429
age10  

## R24 — Sensitivity — the naive ever-vaccinated exposure coding

*Analyses: A36 · Group S3.5*

**Pseudocode**

1. Replace the time-varying protected-time indicator with a fixed ever-vaccinated indicator that treats a physician as exposed for the whole season, including the weeks before vaccination. This is the coding the time-varying definition is designed to avoid, and it is fitted only to show what that avoidance is worth.
2. Fit it unadjusted and adjusted, keeping the frailty and the rest of the specification unchanged, for all-ILI and for the A-cluster.
3. Print the crude rates on the at-risk denominator by ever-vaccinated status, and within vaccinated physicians the rate in protected against pre-protection weeks.
4. Also fit the first-episode-only version of both codings, since restricting to the first episode is where the two codings separate most.
5. Add the physician-level attack-rate contrast under the same fixed exposure, since a naive comparison is often made on that scale rather than on person-time.
6. State the direction of the bias: attributing pre-vaccination person-time to the exposed state moves the all-ILI estimate away from the time-varying result of record, and the physician-level attack-rate contrast moves it further still, because it drops the within-physician timing information entirely.

In [24]:
cat("crude rates on the at-risk denominator\n")
for (g in 0:1) {
  s <- pw[pw$ever_vax == g, ]
  cat(sprintf("  ever vaccinated = %d : %3d episodes / %4d at-risk weeks = %.4f per 100  (%d physicians)\n",
      g, sum(s$event), sum(s$at_risk_new_episode),
      100 * sum(s$event) / sum(s$at_risk_new_episode), length(unique(s$participant_id))))
}
r1 <- sum(pw$event[pw$ever_vax == 1]) / sum(pw$at_risk_new_episode[pw$ever_vax == 1])
r0 <- sum(pw$event[pw$ever_vax == 0]) / sum(pw$at_risk_new_episode[pw$ever_vax == 0])
cat(sprintf("  crude rate ratio = %.4f, naive effectiveness = %.1f%%\n", r1 / r0, 100 * (1 - r1 / r0)))
cat("\nwithin vaccinated physicians' 1,874 weeks\n")
for (g in 0:1) {
  s <- pw[pw$ever_vax == 1 & pw$vax_protected == g, ]
  cat(sprintf("  protected = %d : %3d episodes / %4d at-risk weeks = %.4f per 100\n",
      g, sum(s$event), sum(s$at_risk_new_episode),
      100 * sum(s$event) / sum(s$at_risk_new_episode)))
}
cat("\nmodel fits under the two exposure codings\n")
naive_fit <- function(outcome, x, adj, data, label) {
  d <- data; d$.y <- outcome
  fm <- as.formula(paste("Surv(tstart, tstop, .y) ~", x,
        if (adj) "+ age10 + school_kids_any + ili_freq_ord4" else "",
        "+ frailty(participant_id, distribution = 'gamma')"))
  f <- coxph(fm, data = d, ties = "efron"); b <- f$coefficients[1]; s <- sqrt(diag(f$var))[1]
  cat(sprintf("  %-46s HR = %.4f (%.4f-%.4f) p = %.4f  VE = %5.1f%%\n", label,
      exp(b), exp(b - 1.96 * s), exp(b + 1.96 * s), 2 * pnorm(-abs(b / s)), 100 * (1 - exp(b))))
}
naive_fit(pw$event, "ever_vax", FALSE, pw, "all-ILI, ever vaccinated, unadjusted")
naive_fit(pw$event, "ever_vax", TRUE,  pw, "all-ILI, ever vaccinated, adjusted")
naive_fit(pw$event, "vax_protected", TRUE, pw, "all-ILI, time-varying (of record)")
naive_fit(pw$event_A, "ever_vax", FALSE, pw, "A-cluster, ever vaccinated, unadjusted")
naive_fit(pw$event_A, "ever_vax", TRUE,  pw, "A-cluster, ever vaccinated, adjusted")
naive_fit(pw$event_A, "vax_protected", TRUE, pw, "A-cluster, time-varying (of record)")
cat("\nphysician-level attack-rate contrast under the fixed ever-vaccinated coding\n")
ph_lvl$ever_vax <- unique(pw[, c("participant_id", "ever_vax")])$ever_vax[
  match(ph_lvl$participant_id, unique(pw[, c("participant_id", "ever_vax")])$participant_id)]
ph_lvl$any_ep <- as.integer(ph_lvl$n_ep > 0)
for (adj in c(FALSE, TRUE)) {
  fm <- as.formula(paste("any_ep ~ ever_vax",
        if (adj) "+ age10 + school_kids_any + ili_freq_ord4" else ""))
  f <- glm(fm, data = ph_lvl, family = binomial)
  b <- coef(f)["ever_vax"]; s <- summary(f)$coefficients["ever_vax", 2]
  cat(sprintf("  attack-rate odds ratio, %-10s = %.4f (%.4f-%.4f), one minus the odds ratio = %.1f%%\n",
      ifelse(adj, "adjusted", "unadjusted"), exp(b), exp(b - 1.96 * s), exp(b + 1.96 * s),
      100 * (1 - exp(b))))
}
cat("\nfirst-episode-only panel (", nrow(first_ep), "weeks,", sum(first_ep$event), "events )\n")
for (x in c("ever_vax", "vax_protected")) for (adj in c(FALSE, TRUE)) {
  fm <- as.formula(paste("Surv(tstart, tstop, event) ~", x,
        if (adj) "+ age10 + school_kids_any + ili_freq_ord4" else ""))
  f <- coxph(fm, data = first_ep, ties = "efron"); b <- f$coefficients[1]; s <- sqrt(diag(f$var))[1]
  cat(sprintf("  %-14s %-10s HR = %.4f (%.4f-%.4f) p = %.4f VE = %.1f%%\n", x,
      ifelse(adj, "adjusted", "unadjusted"), exp(b), exp(b - 1.96 * s), exp(b + 1.96 * s),
      2 * pnorm(-abs(b / s)), 100 * (1 - exp(b))))
}

crude rates on the at-risk denominator
  ever vaccinated = 0 : 305 episodes / 2801 at-risk weeks = 10.8890 per 100  (157 physicians)
  ever vaccinated = 1 : 192 episodes / 1825 at-risk weeks = 10.5205 per 100  (91 physicians)
  crude rate ratio = 0.9662, naive effectiveness = 3.4%

within vaccinated physicians' 1,874 weeks
  protected = 0 :  28 episodes /  255 at-risk weeks = 10.9804 per 100
  protected = 1 : 164 episodes / 1570 at-risk weeks = 10.4459 per 100

model fits under the two exposure codings
  all-ILI, ever vaccinated, unadjusted           HR = 0.9437 (0.7290-1.2216) p = 0.6599  VE =   5.6%
  all-ILI, ever vaccinated, adjusted             HR = 0.9014 (0.7013-1.1585) p = 0.4174  VE =   9.9%
  all-ILI, time-varying (of record)              HR = 0.9591 (0.7474-1.2308) p = 0.7429  VE =   4.1%
  A-cluster, ever vaccinated, unadjusted         HR = 0.5387 (0.3197-0.9076) p = 0.0201  VE =  46.1%
  A-cluster, ever vaccinated, adjusted           HR = 0.5250 (0.3132-0.8801) p = 0.0145 

## R25 — Sensitivity — covariate balance by current-season vaccination status

*Analyses: A37 · Group S3.5*

**Pseudocode**

1. Compare the pre-specified baseline covariates between physicians who reported a current-season vaccination and those who did not, at physician level over the reporting cohort.
2. Print the mean or proportion in each group with the group denominators, and the standardized mean difference, computed with the pooled standard deviation so binary and continuous variables are on one scale.
3. This is a balance description, not a test-based screen. It shows which pre-specified adjustments carry weight in the vaccine contrast — particularly the health-condition item and household school-age children.

In [25]:
bal_vars <- c("age10", "female", "university", "sees_pat", "comp_risk_any_nan",
              "school_kids_any", "ili_freq_ord4", "prev_vax", "hh")
out <- data.frame()
for (v in bal_vars) {
  x1 <- rep248[[v]][rep248$vax == 1]; x0 <- rep248[[v]][rep248$vax == 0]
  m1 <- mean(x1, na.rm = TRUE); m0 <- mean(x0, na.rm = TRUE)
  sp <- sqrt((var(x1, na.rm = TRUE) + var(x0, na.rm = TRUE)) / 2)
  out <- rbind(out, data.frame(variable = v, vaccinated = m1, n_vax = sum(!is.na(x1)),
                               unvaccinated = m0, n_unvax = sum(!is.na(x0)),
                               SMD = (m1 - m0) / sp))
}
print(format(out, digits = 4))

           variable vaccinated n_vax unvaccinated n_unvax      SMD
1             age10     4.1527    91       3.9720     157  0.16736
2            female     0.6484    91       0.6752     157 -0.05644
3        university     0.3077    91       0.4904     157 -0.37824
4          sees_pat     0.4725    91       0.4395     157  0.06608
5 comp_risk_any_nan     0.3111    90       0.1645     152  0.34802
6   school_kids_any     0.4945    91       0.3503     157  0.29377
7     ili_freq_ord4     1.4176    91       1.3312     157  0.13363
8          prev_vax     0.7802    91       0.1783     157  1.50267
9                hh     2.6264    91       2.5032     157  0.11162


## R26 — Reproduction recap and session record

*Analyses: — · Group S3.5*

**Pseudocode**

1. Print a compact recap of the quantities this notebook reproduced, so a reader can check agreement without scrolling back through every block.
2. Print the R version and the versions of the estimation packages the models depend on, since a frailty variance and an adaptive-quadrature odds ratio can move in the last decimal place across versions.

In [26]:
recap <- rbind(
 c("cohort and panel",       "filed person-weeks",                sprintf("%d", nrow(pw))),
 c("cohort and panel",       "at-risk person-weeks",              sprintf("%d", sum(pw$at_risk_new_episode))),
 c("cohort and panel",       "incident episodes",                  sprintf("%d", sum(pw$event))),
 c("cohort and panel",       "A-cluster episodes",                 sprintf("%d", sum(pw$event_A))),
 c("cohort and panel",       "incidence per 100 at-risk weeks",    sprintf("%.4f", 100*sum(pw$event)/sum(pw$at_risk_new_episode))),
 c("M5 primary incidence",   "vaccination HR",                     sprintf("%.4f", exp(fit_M5$coefficients[1]))),
 c("M5 primary incidence",   "age HR per ten years",               sprintf("%.4f", exp(fit_M5$coefficients[2]))),
 c("M5 primary incidence",   "household school-age children HR",   sprintf("%.4f", exp(fit_M5$coefficients[3]))),
 c("M5 primary incidence",   "susceptibility HR",                  sprintf("%.4f", exp(fit_M5$coefficients[4]))),
 c("M5 primary incidence",   "frailty variance theta",             sprintf("%.4f", fit_M5$history[[1]]$theta)),
 c("M6 vaccine",             "A-cluster HR",                       sprintf("%.4f", exp(r_A["b"]))),
 c("M6 vaccine",             "A-cluster se(logHR)",                sprintf("%.4f", r_A["se"])),
 c("M6 vaccine",             "A-cluster effectiveness",            sprintf("%.1f%%", 100*(1-exp(r_A["b"])))),
 c("M6 vaccine",             "B-cluster negative control HR",      sprintf("%.4f", exp(r_B["b"]))),
 c("M6 vaccine",             "protected person-weeks",             sprintf("%d", sum(pw$vax_protected))),
 c("M1 uptake",              "apparent AUC",                       sprintf("%.4f", auc_of(fit_M1$y, fit_M1$fitted.values))),
 c("M2 susceptibility",      "age cumulative OR",                  sprintf("%.4f", exp(coef(fit_M2)["age10"]))),
 c("M3 care-seeking",        "susceptibility OR",                  sprintf("%.4f", exp(coef(fit_M3)["ili_freq_ord4"]))),
 c("M7a gradient",           "IRR per ten more reported weeks",    sprintf("%.4f", exp(coef(fit_grad)["nwk10"]))),
 c("M7b forward",            "conditional OR",                     sprintf("%.4f", exp(fixef(fit_fwd)["symptomatic"]))),
 c("M7b backward",           "conditional OR",                     sprintf("%.4f", exp(fixef(fit_bwd)["gap_before"]))),
 c("M7b backward",           "marginal GEE OR",                    sprintf("%.4f", exp(coef(fit_gee)["gap_before"]))),
 c("M8 occupational",        "aerosol-share HR",                   sprintf("%.4f", exp(fit_M8$coefficients["aerosol_ord"]))),
 c("M8 occupational",        "mask-use HR",                        sprintf("%.4f", exp(fit_M8$coefficients["mask"]))))
colnames(recap) <- c("model", "quantity", "value")
print(as.data.frame(recap), row.names = FALSE)

cat("\n")
cat("R version :", R.version.string, "\n")
for (p in c("survival", "MASS", "lme4", "geepack", "nnet", "pROC", "readxl"))
  cat(sprintf("  %-10s %s\n", p, as.character(packageVersion(p))))

                model                         quantity   value
     cohort and panel               filed person-weeks    4729
     cohort and panel             at-risk person-weeks    4626
     cohort and panel                incident episodes     497
     cohort and panel               A-cluster episodes      91
     cohort and panel  incidence per 100 at-risk weeks 10.7436
 M5 primary incidence                   vaccination HR  0.9591
 M5 primary incidence             age HR per ten years  0.8513
 M5 primary incidence household school-age children HR  1.3633
 M5 primary incidence                susceptibility HR  1.3576
 M5 primary incidence           frailty variance theta  0.3332
           M6 vaccine                     A-cluster HR  0.5368
           M6 vaccine              A-cluster se(logHR)  0.2772
           M6 vaccine          A-cluster effectiveness   46.3%
           M6 vaccine    B-cluster negative control HR  1.0783
           M6 vaccine           protected person-weeks 